By moving from a 770M Encoder-Decoder to a 7B Decoder-only model like CodeQwen-1.5, you aren't just getting more "memory," you're getting a massive upgrade in reasoning density.
Why CodeQwen will crush the Zero-Shot Barrier

    Instruction Following: CodeT5+ was trained to predict "the next chunk of code." CodeQwen was trained to follow instructions. It understands that when you say "Context:...", that DDL is a constraint it must follow, not just a hint.

    Vocabulary Depth: Your 770M model failed WikiSQL because it didn't recognize "messy" headers. CodeQwen has been pre-trained on billions of lines of real-world GitHub code where headers are always messy. It has a much stronger "semantic bridge" between a question like "What is the nationality?" and a column named Nationality (Citizenship).

    The "Table ()" Fix: The strange syntax errors you saw (table (), table table") are symptoms of a small model "panicking" because it doesn't truly understand SQL grammar—it only knows patterns. A 7B model has a much deeper internal representation of SQL grammar.

In [22]:
!pip install -q --force-reinstall --no-cache-dir huggingface_hub transformers datasets trl peft accelerate bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 280.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 127.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 294.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 258.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.6/79.6 kB 311.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.8/647.8 kB 203.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 259.0 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 103.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.6/721.6 kB 156.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 120.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 361.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 160.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━

In [ ]:
# Install the latest versions of the PEFT and TRL libraries
!pip install -q -U bitsandbytes peft accelerate transformers trl datasets huggingface_hub torchao

# Model Loading (4-bit Quantization)

We use bitsandbytes to load the model in 4-bit NormalFloat (NF4). This reduces the memory footprint from ~28GB to approximately 5.5GB.

In [ ]:
import torch
import gc
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# 1. Ensure VRAM is empty
gc.collect()
torch.cuda.empty_cache()

model_id = "Qwen/CodeQwen1.5-7B-Chat"

# 2. Configure 4-bit Quantization (Strictly Float16 for T4 compatibility)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

# 3. Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# 4. Load Model (Strictly Float16)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map={"": 0},
    torch_dtype=torch.float16,
    trust_remote_code=True
)

print("Model loaded successfully!")

# Data Preprocessing (The Chat Template)

Unlike T5, Decoder-only models perform best when instructions are formatted as a dialogue. We will map the b-mc2 dataset into the Qwen Chat Template.

In [ ]:
from datasets import load_dataset

# Load first 20k rows for initial training
dataset = load_dataset("b-mc2/sql-create-context", split="train[:20000]")

def format_instruction(sample):
    messages = [
        {"role": "system", "content": "You are a SQL expert. Use the provided DDL context to write a correct SQL query for the question."},
        {"role": "user", "content": f"Context: {sample['context']}\n\nQuestion: {sample['question']}"},
        {"role": "assistant", "content": f"{sample['answer']}"}
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

dataset = dataset.map(format_instruction)
print(f"Sample formatted text:\n{dataset[0]['text']}")

# LoRA Configuration

We will only train 0.5% - 1% of the model's parameters (the "adapters"). This prevents the catastrophic forgetting of its pre-trained SQL knowledge.

In [ ]:
# import torch
# import gc
# from transformers import AutoTokenizer, AutoModelForCausalLM
# from peft import get_peft_model, LoraConfig
# from trl import SFTTrainer, SFTConfig

# # 1. Total VRAM Wipe
# gc.collect()
# torch.cuda.empty_cache()

# model_id = "Qwen/CodeQwen1.5-7B-Chat"

# # 2. Tokenizer
# tokenizer = AutoTokenizer.from_pretrained(model_id)
# tokenizer.pad_token = tokenizer.eos_token

# # 3. Distributed Load
# model = AutoModelForCausalLM.from_pretrained(
#     model_id,
#     device_map="auto",            # Splits the model across both T4 GPUs
#     torch_dtype=torch.float16,  
#     trust_remote_code=True
# )

# # 4. Silence the internal config
# model.config.torch_dtype = torch.float16
# model.config.bf16 = False         # Explicitly disable BF16 deep in the config

# # 5. Apply LoRA 
# peft_config = LoraConfig(
#     r=16,
#     lora_alpha=32,
#     target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
#     lora_dropout=0.05,
#     bias="none",
#     task_type="CAUSAL_LM"
# )
# model = get_peft_model(model, peft_config)

# #THE ULTIMATE OVERRIDE
# # This catches any dynamically created buffers (like Qwen's RoPE embeddings)
# # or PEFT adapters and forces them all to float16, preventing the GradScaler crash.
# model = model.to(dtype=torch.float16)

# # 6. Distributed Training Configuration
# sft_config = SFTConfig(
#     output_dir="./codeqwen-7b-sql-v2",
#     dataset_text_field="text",
#     max_length=512,
#     per_device_train_batch_size=2,   # 2 per GPU = 4 total per step
#     gradient_accumulation_steps=4,   # Effective batch size = 16
#     learning_rate=2e-4,
#     num_train_epochs=1,
#     logging_steps=10,
#     fp16=True,                       # GradScaler will safely process Float16
#     bf16=False,     
#     optim="paged_adamw_8bit",        # 8-bit optimizer to keep VRAM extra safe
#     gradient_checkpointing=True,     
#     report_to="none"
# )

# # 7. Launch Trainer
# trainer = SFTTrainer(
#     model=model,
#     train_dataset=dataset,
#     args=sft_config
# )

# trainer.train()

In [3]:
import torch
import gc

# Delete any old model/trainer variables if they exist
if 'model' in globals(): del model
if 'trainer' in globals(): del trainer
if 'tokenizer' in globals(): del tokenizer

# Force garbage collection and empty VRAM cache
gc.collect()
torch.cuda.empty_cache()

# Optional: Print how much VRAM is free now
free_vram = torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_reserved(0)
print(f"Free VRAM: {free_vram / 1024**3:.2f} GB")

Free VRAM: 0.13 GB


# Training

The SFTTrainer (Supervised Fine-Tuning Trainer) handles the memory-efficient training loop.

In [ ]:
import torch
import gc
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import get_peft_model, LoraConfig
from trl import SFTTrainer, SFTConfig

# 1. Total VRAM Wipe
gc.collect()
torch.cuda.empty_cache()

print(f"GPUs available: {torch.cuda.device_count()}")

model_id = "Qwen/CodeQwen1.5-7B-Chat"

# 2. Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

# 3. PURE FLOAT16 LOAD (No Quantization)
# By removing BitsAndBytes, we completely avoid the BFloat16 bug.
# device_map="auto" seamlessly splits the 14GB weights across both 16GB GPUs.
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float16,  
    trust_remote_code=True
)

# Force the config to ensure SFTTrainer behaves
model.config.torch_dtype = torch.float16

# 4. Apply LoRA 
# Notice we no longer need prepare_model_for_kbit_training()
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, peft_config)

# 5. Distributed Training Configuration
# 6. Distributed Training Configuration
sft_config = SFTConfig(
    output_dir="./codeqwen-7b-sql-v2",
    dataset_text_field="text",
    max_length=512,
    per_device_train_batch_size=1,   # DROPPED TO 1 (Saves massive logit VRAM)
    gradient_accumulation_steps=8,   # INCREASED TO 8
    learning_rate=2e-4,
    num_train_epochs=1,
    logging_steps=10,
    fp16=True,                       
    bf16=False,     
    optim="paged_adamw_8bit",        # Ensure this is set to 8-bit to page memory to CPU if needed
    gradient_checkpointing=True,     
    report_to="none"
)

# 7. Launch Trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=sft_config
)

trainer.train()

# Inference (Zero-Shot Capability)

After training, we test the model using the exact same chat format. This is where the model's reasoning really shines.

In [ ]:
def generate_sql(question, context):
    messages = [
        {"role": "system", "content": "You are a SQL expert. Use the provided DDL context to write a correct SQL query for the question."},
        {"role": "user", "content": f"Context: {context}\n\nQuestion: {question}"}
    ]
    
    prompt = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    
    # Explicitly set the terminators to Qwen's ChatML stop tokens
    # We ignore ID 4 if it's causing the "SELECT" short-circuit
    stop_tokens = ["<|im_end|>", "<|endoftext|>"]
    stop_token_ids = [tokenizer.convert_tokens_to_ids(t) for t in stop_tokens]

    with torch.no_grad():
        outputs = model.generate(
            **inputs, 
            max_new_tokens=256,   # Increased to give it room for long queries
            do_sample=False, 
            # Force the model to use the proper stop tokens
            eos_token_id=stop_token_ids, 
            pad_token_id=tokenizer.eos_token_id
        )
    
    # Slice the output to get only the new tokens
    generated_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    decoded = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    
    return decoded.strip()

# Test Case
print(f"Predicted SQL: {generate_sql(question_sample, context_sample)}")

In [ ]:
if hasattr(model, "active_adapter"):
    print(f"Adapter Active: {model.active_adapter}")
else:
    print("WARNING: No adapters found. You are running the base model!")
    # To fix, run: 
    # from peft import PeftModel
    # model = PeftModel.from_pretrained(model, "./codeqwen-7b-sql-final")

In [ ]:
def debug_generate_sql(question, context):
    messages = [
        {"role": "system", "content": "You are a SQL expert. Use the provided DDL context to write a correct SQL query for the question."},
        {"role": "user", "content": f"Context: {context}\n\nQuestion: {question}"}
    ]
    prompt = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs, 
            max_new_tokens=128,
            do_sample=True,      # 🔴 Use sampling to break the "SELECT" loop
            temperature=0.7,     # 🔴 Add some "creativity"
            # 🔴 BAN THE EOS TOKEN (ID 4)
            bad_words_ids=[[4]], 
            pad_token_id=tokenizer.eos_token_id
        )
    
    generated_ids = outputs[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=False).strip()

print(f"Debug SQL: {debug_generate_sql(question_sample, context_sample)}")

In [ ]:
# The Naked Truth Diagnostic Script
context_sample = "CREATE TABLE stadium (stadium_id NUMBER, name TEXT, capacity NUMBER)"
question_sample = "How many stadiums have a capacity greater than 50000?"

messages = [
    {"role": "system", "content": "You are a SQL expert. Use the provided DDL context to write a correct SQL query for the question."},
    {"role": "user", "content": f"Context: {context_sample}\n\nQuestion: {question_sample}"}
]

# 1. Let's see EXACTLY what string the template is building
prompt = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
print("--- 1. EXACT PROMPT FED TO MODEL ---")
print(repr(prompt))

# 2. Generate with ZERO restrictions (No stop tokens, no sampling, just raw output)
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(
        inputs.input_ids, 
        max_new_tokens=50,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )
    
# 3. Isolate the newly generated tokens
generated_only = outputs[0][inputs.input_ids.shape[1]:]

print("\n--- 2. RAW TOKEN IDs PRODUCED ---")
print(generated_only.tolist())

print("\n--- 3. RAW DECODED STRING ---")
print(repr(tokenizer.decode(generated_only)))

In [1]:
import torch
import gc
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

# 1. Total VRAM Wipe
gc.collect()
torch.cuda.empty_cache()

# 2. Paths
model_id = "Qwen/CodeQwen1.5-7B-Chat"
adapter_path = "./codeqwen-7b-sql-final" # The folder you saved your weights to

print("Loading Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id)

print("Loading Base Model strictly to GPU 0...")
# By forcing device_map={"": 0}, we guarantee no parts spill to the CPU
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map={"": 0}, 
    torch_dtype=torch.float16,
)

print("Attaching TRAINED Adapters...")
model = PeftModel.from_pretrained(base_model, adapter_path)
model.eval() # Lock the weights for inference

# The EOS Fix
proper_eos_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
model.config.eos_token_id = proper_eos_id
model.generation_config.eos_token_id = proper_eos_id

print("Ready for Inference!\n")

def final_generate_sql(question, context):
    messages = [
        {"role": "system", "content": "You are a SQL expert. Use the provided DDL context to write a correct SQL query for the question."},
        {"role": "user", "content": f"Context: {context}\n\nQuestion: {question}"}
    ]
    
    prompt = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    
    # Ensure inputs go to the exact same device as the model (cuda:0)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda:0")
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs, 
            max_new_tokens=128,
            do_sample=False,
            eos_token_id=proper_eos_id,
            pad_token_id=tokenizer.eos_token_id
        )
    
    generated_ids = outputs[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

# Let's test the trained brain!
context_sample = "CREATE TABLE stadium (stadium_id NUMBER, name TEXT, capacity NUMBER)"
question_sample = "How many stadiums have a capacity greater than 50000?"

print(f"Final Predicted SQL: {final_generate_sql(question_sample, context_sample)}")

Loading Tokenizer...


config.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/972 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/1.42M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/387 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/212 [00:00<?, ?B/s]

Attaching TRAINED Adapters...


/usr/local/lib/python3.12/dist-packages/peft/config.py:220: UserWarning: Unexpected keyword arguments ['lora_ga_config', 'use_bdlora'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(
The following generation flags are not valid and may be ignored: ['top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Ready for Inference!

Final Predicted SQL: SELECT COUNT(*) FROM stadium WHERE capacity > 50000


# Save and Export

We save the adapters (the weights we trained) and the tokenizer.

In [ ]:
# Save the LoRA adapters
model.save_pretrained("./final_sql_adapter")
tokenizer.save_pretrained("./final_sql_adapter")

print("Training Complete. Adapters saved to ./final_sql_adapter")

In [ ]:
import shutil
import os

# 1. Save the trained adapters and tokenizer to a folder
model_path = "./codeqwen-7b-sql-final"
model.save_pretrained(model_path)
tokenizer.save_pretrained(model_path)

# 2. Zip the folder
shutil.make_archive("codeqwen_7b_sql_adapters", 'zip', model_path)

print("Zip file created: codeqwen_7b_sql_adapters.zip")

# Evaluation and Assessment

Let's use the Exact Match and the Proximal Match metrics to evaluate the model

In [1]:
import torch
import json
import re
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from datasets import load_dataset

# ==========================================
# 1. METRICS & NORMALIZATION
# ==========================================
def normalize_sql(sql):
    """Removes extra whitespace, converts to lowercase, and strips semicolons."""
    sql = str(sql).lower().strip()
    sql = re.sub(r'\s+', ' ', sql)
    if sql.endswith(';'):
        sql = sql[:-1]
    return sql.strip()

def calculate_exact_match(pred, target):
    return 1 if normalize_sql(pred) == normalize_sql(target) else 0

def calculate_proximal_match(pred, target):
    """Calculates Jaccard Similarity (Token Overlap)."""
    pred_tokens = set(normalize_sql(pred).split())
    target_tokens = set(normalize_sql(target).split())
    
    if not target_tokens: return 0.0
    
    intersection = pred_tokens.intersection(target_tokens)
    union = pred_tokens.union(target_tokens)
    return len(intersection) / len(union)

# ==========================================
# 2. MODEL LOADING (Multi-GPU)
# ==========================================
model_id = "Qwen/CodeQwen1.5-7B-Chat"
adapter_path = "./codeqwen-7b-sql-final"

print("Loading tokenizer and model across BOTH GPUs...")
tokenizer = AutoTokenizer.from_pretrained(model_id)

base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",         # 🔴 USE BOTH GPUS (7GB each)
    torch_dtype=torch.float16,
)
model = PeftModel.from_pretrained(base_model, adapter_path)
model.eval()

# EOS token fix
proper_eos_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
model.config.eos_token_id = proper_eos_id
model.generation_config.eos_token_id = proper_eos_id

# ==========================================
# 3. INFERENCE PIPELINE
# ==========================================
def predict_sql(question, context):
    messages = [
        {"role": "system", "content": "You are a SQL expert. Use the provided DDL context to write a correct SQL query for the question."},
        {"role": "user", "content": f"Context: {context}\n\nQuestion: {question}"}
    ]
    prompt = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    
    # 🔴 Dynamically send inputs to whatever device the first layer is on
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs, 
            max_new_tokens=128,
            do_sample=False,
            eos_token_id=proper_eos_id,
            pad_token_id=tokenizer.eos_token_id
        )
    
    generated_ids = outputs[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

# ==========================================
# 4. DATASET EVALUATION RUNNER
# ==========================================
def evaluate_dataset(dataset_name, data, MAX_SAMPLES=100):
    print(f"\n--- Evaluating on {dataset_name} (Samples: {MAX_SAMPLES if MAX_SAMPLES else 'ALL'}) ---")
    
    total_em = 0
    total_proximal = 0
    count = 0
    
    for item in tqdm(data[:MAX_SAMPLES]):
        question = item.get('question', '')
        context = item.get('context', item.get('schema', '')) 
        target_sql = item.get('answer', item.get('query', ''))
        
        predicted_sql = predict_sql(question, context)
        
        em = calculate_exact_match(predicted_sql, target_sql)
        proximal = calculate_proximal_match(predicted_sql, target_sql)
        
        total_em += em
        total_proximal += proximal
        count += 1

    avg_em = (total_em / count) * 100
    avg_proximal = (total_proximal / count) * 100
    
    print(f"\nResults for {dataset_name}:")
    print(f"Exact Match (EM): {avg_em:.2f}%")
    print(f"Proximal Match:   {avg_proximal:.2f}%")

# ==========================================
# 5. EXECUTE
# ==========================================
MAX_TEST_SIZE = 100 # Change to None when you're ready for the full run!

# 1. WikiSQL Test Data
try:
    with open('/kaggle/input/wikisql/wikisql_test.json', 'r') as f:
        holdout_data = json.load(f)
    evaluate_dataset("Holdout Training Data", holdout_data, MAX_SAMPLES=MAX_TEST_SIZE)
except Exception as e:
    print(f"Could not load holdout data: {e}. Check the file path.")

# 2. Spider Dataset
print("\nDownloading Spider Dataset...")
spider_dataset = load_dataset("spider", split="validation")
spider_formatted = []
for row in spider_dataset:
    spider_formatted.append({
        "question": row["question"],
        "context": f"-- DB: {row['db_id']}",
        "answer": row["query"]
    })

evaluate_dataset("Spider Validation Set", spider_formatted, MAX_SAMPLES=MAX_TEST_SIZE)

Loading tokenizer and model across BOTH GPUs...


config.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/972 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/1.42M [00:00<?, ?B/s]

...

Generating train split:   0%|          | 0/7000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1034 [00:00<?, ? examples/s]


--- Evaluating on Spider Validation Set (Samples: 100) ---



  0%|          | 0/100 [00:00<?, ?it/s]The following generation flags are not valid and may be ignored: ['top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.

100%|██████████| 100/100 [04:41<00:00,  2.82s/it]


Results for Spider Validation Set:
Exact Match (EM): 12.00%
Proximal Match:   50.89%


# Sanity Check

In [2]:
import random

# Ensure we are using the formatted spider data from the previous step
# If you didn't run the evaluation cell, make sure spider_formatted is defined
samples = random.sample(spider_formatted, 10)

print(f"{'='*30} SPIDER RANDOM INFERENCE (10 SAMPLES) {'='*30}\n")

for i, item in enumerate(samples):
    question = item['question']
    context = item['context']
    ground_truth = item['answer']
    
    # Run the model!
    predicted = predict_sql(question, context)
    
    print(f"SAMPLE #{i+1}")
    print(f"QUESTION:  {question}")
    print(f"CONTEXT:   {context}")
    print(f"GROUND TRUTH: {ground_truth}")
    print(f"PREDICTED:    {predicted}")
    
    # Simple visual indicator of match
    is_exact = normalize_sql(predicted) == normalize_sql(ground_truth)
    match_status = "EXACT MATCH" if is_exact else "⚠️ STRING MISMATCH (Check Logic)"
    print(f"STATUS:    {match_status}")
    print("-" * 80 + "\n")

============================== SPIDER RANDOM INFERENCE (10 SAMPLES) ==============================

SAMPLE #1
QUESTION:  What is the most populace city that speaks English?
CONTEXT:   -- DB: world_1
GROUND TRUTH: SELECT T1.Name ,  T1.Population FROM city AS T1 JOIN countrylanguage AS T2 ON T1.CountryCode  =  T2.CountryCode WHERE T2.Language  =  "English" ORDER BY T1.Population DESC LIMIT 1
PREDICTED:    SELECT Name FROM city WHERE Population = (SELECT MAX(Population) FROM city WHERE Language = "English")
STATUS:    ⚠️ STRING MISMATCH (Check Logic)
--------------------------------------------------------------------------------

SAMPLE #2
QUESTION:  What are the different template type codes, and how many documents use each type?
CONTEXT:   -- DB: cre_Doc_Template_Mgt
GROUND TRUTH: SELECT T1.template_type_code ,  count(*) FROM Templates AS T1 JOIN Documents AS T2 ON T1.template_id  =  T2.template_id GROUP BY T1.template_type_code
PREDICTED:    SELECT Template_Type_Code, COUNT(*) FROM Do

The model isn't "failing" the logic; it’s actually suffering from "Schema Blindness." Let's append the actual DDL context before inference.

# The Schema Parser

In [7]:
import json
import os


spider_path = "/kaggle/input/datasets/jeromeblanchet/yale-universitys-spider-10-nlp-dataset/spider"
tables_file = os.path.join(spider_path, "tables.json")

if os.path.exists(tables_file):
    with open(tables_file, 'r') as f:
        spider_tables = json.load(f)
    print(f"Successfully loaded {len(spider_tables)} database schemas from Kaggle.")
else:
    print("Could not find tables.json. Please check the dataset name in your Kaggle sidebar.")

Successfully loaded 166 database schemas from Kaggle.


In [8]:
def get_spider_schema(db_id, tables_list):
    """Formats the schema metadata into a DDL prompt context."""
    for db in tables_list:
        if db['db_id'] == db_id:
            table_names = db['table_names_original']
            column_names = db['column_names_original'] 
            column_types = db['column_types']
            
            ddl_parts = []
            for i, table_name in enumerate(table_names):
                # Format: "column_name TYPE, column_name TYPE"
                cols = [f"{column_names[j][1]} {column_types[j].upper()}" 
                        for j, (t_idx, _) in enumerate(column_names) if t_idx == i]
                ddl_parts.append(f"CREATE TABLE {table_name} ({', '.join(cols)})")
            return " ".join(ddl_parts)
    return ""

def evaluate_spider_schema_aware(data, tables_list, MAX_SAMPLES=100):
    print(f"\n--- Running Schema-Aware Evaluation ({MAX_SAMPLES} samples) ---")
    total_em = 0
    total_proximal = 0
    
    for item in tqdm(data[:MAX_SAMPLES]):
        db_id = item['db_id']
        question = item['question']
        target_sql = item['query']
        
        # THE MAGIC STEP: Fetch actual DDL
        context = get_spider_schema(db_id, tables_list)
        
        # Predict using your model
        predicted_sql = final_generate_sql(question, context)
        
        em = calculate_exact_match(predicted_sql, target_sql)
        proximal = calculate_proximal_match(predicted_sql, target_sql)
        
        total_em += em
        total_proximal += proximal

    avg_em = (total_em / MAX_SAMPLES) * 100
    avg_proximal = (total_proximal / MAX_SAMPLES) * 100
    
    print(f"\nFINAL RESULTS (Schema-Aware):")
    print(f"Exact Match (EM): {avg_em:.2f}%")
    print(f"Proximal Match:   {avg_proximal:.2f}%")

# Run it!

# Improved Sanity Check
Now, let's run those same 10 random samples, but this time we'll give the model the full DDL. Watch how the "Predicted SQL" becomes much more precise.

In [9]:
# Assuming 'spider_dataset' and 'spider_tables' are loaded
samples = random.sample(list(spider_dataset), 5) # Let's do 5 high-quality checks

print(f"{'='*30} SPIDER DDL-AWARE INFERENCE {'='*30}\n")

for i, row in enumerate(samples):
    db_id = row['db_id']
    question = row['question']
    ground_truth = row['query']
    
    # THE KEY CHANGE: Get real DDL instead of just the DB Name
    context = get_spider_schema(db_id, spider_tables)
    
    predicted = predict_sql(question, context) # Using your existing function
    
    print(f"SAMPLE #{i+1} [DB: {db_id}]")
    print(f"QUESTION:  {question}")
    print(f"DDL PROVIDED: {context[:100]}...") # Printing just a snippet
    print(f"GROUND TRUTH: {ground_truth}")
    print(f"PREDICTED:    {predicted}")
    
    is_exact = normalize_sql(predicted) == normalize_sql(ground_truth)
    print(f"STATUS:    {' EXACT MATCH' if is_exact else 'MISMATCH'}")
    print("-" * 80 + "\n")

============================== SPIDER DDL-AWARE INFERENCE ==============================

SAMPLE #1 [DB: tvshow]
QUESTION:  What is the title of all the cartools that are on the TV Channel with the series name "Sky Radio"?
DDL PROVIDED: CREATE TABLE TV_Channel (id TEXT, series_name TEXT, Country TEXT, Language TEXT, Content TEXT, Pixel...
GROUND TRUTH: SELECT T2.Title FROM TV_Channel AS T1 JOIN Cartoon AS T2 ON T1.id = T2.Channel WHERE T1.series_name = "Sky Radio";
PREDICTED:    SELECT T1.Title FROM Cartoon AS T1 JOIN TV_Channel AS T2 ON T1.Channel = T2.id WHERE T2.series_name = "Sky Radio"
STATUS:    MISMATCH
--------------------------------------------------------------------------------

SAMPLE #2 [DB: employee_hire_evaluation]
QUESTION:  What are the minimum and maximum number of products across all the shops?
DDL PROVIDED: CREATE TABLE employee (Employee_ID NUMBER, Name TEXT, Age NUMBER, City TEXT) CREATE TABLE shop (Shop...
GROUND TRUTH: SELECT min(Number_products) ,  max(Number_

Sample #3, #4, and #5 reveal an interesting phenomenon: Instructional Hallucination (Over-Inference).

The model is now SQL-Literate and Schema-Aware, which is a huge leap from where we started. It knows how to join, it knows how to aggregate, and it respects the DDL.

The "Hallucination" issue is actually a sign of a strong adapter—it's just a bit too eager to provide the "complete" answer it remembers from training. In the industry, we call this a lack of instruction following robustness.

# How do we fix the "Extra" Filters

    Lower the Epochs: You might have trained a bit too long for this specific dataset size, causing the model to memorize samples.

    Add Negative Samples: Train the model on questions where it must ignore certain columns.

    Inference-time Prompting: You can add a line to your System Prompt: "Strictly output only the logic requested in the question. Do not add additional filters or constraints."

Let's classify errors into 3 buckets:

    Type A: Syntactic Variations (Case, Alias, Order) — Model succeeded logically.

    Type B: Over-Inference (Adding extra filters) — Model over-fitted to training data.

    Type C: Exact Match — Successfully captured the prompt intent.

# Negative Sampling & Data Augmentation

This function modifies the training data to include "Distractor Schemas." By injecting unrelated table definitions into the context, we train the model to ignore irrelevant information and only use columns necessary for the user's specific question.

In [1]:
import json
import os
import random
from datasets import load_dataset

# 1. Define the path to Spider's tables.json on Kaggle
spider_path = "/kaggle/input/datasets/jeromeblanchet/yale-universitys-spider-10-nlp-dataset/spider" 
tables_file = os.path.join(spider_path, "tables.json")

# 2. Load the metadata into 'spider_tables'
if os.path.exists(tables_file):
    with open(tables_file, 'r') as f:
        spider_tables = json.load(f)
    print(f"Successfully loaded metadata for {len(spider_tables)} databases.")
else:
    # Fallback if you are using a different Kaggle dataset name
    print("tables.json not found. Searching for any tables.json in /kaggle/input...")
    import glob
    found_files = glob.glob("/kaggle/input/**/tables.json", recursive=True)
    if found_files:
        with open(found_files[0], 'r') as f:
            spider_tables = json.load(f)
        print(f"Found and loaded: {found_files[0]}")
    else:
        raise FileNotFoundError("Could not find tables.json. Please ensure the Spider dataset is added to your Kaggle notebook.")

# 3. Define the Injection Function (re-stating for safety)
def inject_negative_samples(example, all_tables, distractor_count=2):
    current_db = example['db_id']
    distractors = [db for db in all_tables if db['db_id'] != current_db]
    selected_distractors = random.sample(distractors, min(distractor_count, len(distractors)))
    
    distractor_ddl = ""
    for db in selected_distractors:
        idx = random.randint(0, len(db['table_names_original']) - 1)
        table_name = db['table_names_original'][idx]
        cols = [db['column_names_original'][j][1] for j, t_idx in enumerate(db['column_names_original']) if t_idx[0] == idx]
        distractor_ddl += f" CREATE TABLE {table_name} ({', '.join(cols[:3])})"
        
    # We create a new 'context' field for the prompt
    # Note: If your dataset doesn't have a 'context' field, we create it from the db_id logic
    example['context'] = example.get('context', f"-- DB: {current_db}") + distractor_ddl
    return example

# 4. Map the Dataset
dataset = load_dataset("spider", split="train")
dataset = dataset.map(lambda x: inject_negative_samples(x, spider_tables), batched=False)

print(f"🚀 Dataset ready with negative samples! Samples: {len(dataset)}")

Successfully loaded metadata for 166 databases.


README.md: 0.00B [00:00, ?B/s]

spider/train-00000-of-00001.parquet:   0%|          | 0.00/831k [00:00<?, ?B/s]

spider/validation-00000-of-00001.parquet:   0%|          | 0.00/126k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1034 [00:00<?, ? examples/s]

Map:   0%|          | 0/7000 [00:00<?, ? examples/s]

🚀 Dataset ready with negative samples! Samples: 7000


# Regularized PEFT Configuration

We are applying Dropout and Rank Bottlenecking to prevent overfitting. By setting r=16, we limit the adapter's capacity, forcing it to learn the general grammar of SQL instead of specific training rows.

In [3]:
# Force upgrade specifically to a version > 0.46.1
!pip install -q -U bitsandbytes>=0.46.1 accelerate transformers trl datasets
# !pip install -q -U huggingface_hub datasets
# !pip install --force-reinstall --no-cache-dir huggingface_hub transformers

In [6]:
# 1. Surgically uninstall everything related to our stack
!pip uninstall -y transformers huggingface_hub peft trl datasets bitsandbytes accelerate

# 2. Install the absolute latest versions together so they sync perfectly
!pip install -q -U transformers huggingface_hub peft trl datasets bitsandbytes accelerate --no-cache-dir

Found existing installation: transformers 5.7.0
Uninstalling transformers-5.7.0:
  Successfully uninstalled transformers-5.7.0
Found existing installation: huggingface_hub 1.12.2
Uninstalling huggingface_hub-1.12.2:
  Successfully uninstalled huggingface_hub-1.12.2
Found existing installation: peft 0.18.1
Uninstalling peft-0.18.1:
  Successfully uninstalled peft-0.18.1
Found existing installation: trl 1.3.0
Uninstalling trl-1.3.0:
  Successfully uninstalled trl-1.3.0
Found existing installation: datasets 4.8.5
Uninstalling datasets-4.8.5:
  Successfully uninstalled datasets-4.8.5
Found existing installation: bitsandbytes 0.49.2
Uninstalling bitsandbytes-0.49.2:
  Successfully uninstalled bitsandbytes-0.49.2
Found existing installation: accelerate 1.13.0
Uninstalling accelerate-1.13.0:
  Successfully uninstalled accelerate-1.13.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 119.0 MB/s eta 0:00:000:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.8/647.8 kB 380.6 MB/s eta

In [2]:
# from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, EarlyStoppingCallback
# from peft import LoraConfig
# from trl import SFTTrainer
# import torch

# # 1. Dataset Splitting
# dataset_split = dataset.train_test_split(test_size=0.1)
# train_data = dataset_split["train"]
# eval_data = dataset_split["test"]

# # THE FIX: ACTUALLY LOAD A FRESH BASE MODEL
# model_id = "Qwen/CodeQwen1.5-7B-Chat"

# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.float16,
# )

# print("Loading a truly fresh base model...")
# base_model = AutoModelForCausalLM.from_pretrained(
#     model_id,
#     quantization_config=bnb_config,
#     device_map="auto",
#     torch_dtype=torch.float16,  # Avoids the BFloat16 crash!
# )

# tokenizer = AutoTokenizer.from_pretrained(model_id)
# tokenizer.pad_token = tokenizer.eos_token
# # --------------------------------------------------

# # 2. LoRA Config
# peft_config = LoraConfig(
#     r=16, 
#     lora_alpha=32,
#     target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
#     lora_dropout=0.15,
#     bias="none",
#     task_type="CAUSAL_LM",
# )

# # 3. Training Arguments
# training_args = TrainingArguments(
#     output_dir="./codeqwen-sql-v2-precision",
#     per_device_train_batch_size=1,
#     gradient_accumulation_steps=8,
#     learning_rate=2e-5,
#     max_steps=1200,
#     logging_steps=10,
#     eval_strategy="steps", 
#     eval_steps=200,          
#     save_strategy="steps",
#     save_steps=200,          
#     save_total_limit=2,
#     load_best_model_at_end=True,
#     fp16=True,
#     report_to="none"
# )

# # 4. Corrected Formatting Function
# def formatting_prompts_func(example):
#     return (
#         f"<|im_start|>system\n"
#         f"You are a SQL expert. Strictly output ONLY the SQL query requested. "
#         f"Do NOT include additional columns, filters, or JOINs not explicitly required by the question.<|im_end|>\n"
#         f"<|im_start|>user\n"
#         f"Context: {example['context']}\n\n"
#         f"Question: {example['question']}<|im_end|>\n"
#         f"<|im_start|>assistant\n"
#         f"{example['query']}<|im_end|>"
#     )

# # 5. Initialize Trainer
# trainer = SFTTrainer(
#     model=base_model,            #  Now using the freshly loaded model
#     train_dataset=train_data,
#     eval_dataset=eval_data,      
#     peft_config=peft_config,
#     formatting_func=formatting_prompts_func,
#     args=training_args,
#     callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
# )

# print(f"Training with {len(train_data)} train and {len(eval_data)} eval samples...")
# trainer.train()

ModuleNotFoundError: No module named 'trl'

# Training with Validation Guardrails

We reduce the total steps and introduce Early Stopping. If the model stops improving on the validation set, we kill the process to preserve generalization.

In [8]:
import torch
import gc

# 1. Clear Python's garbage collector
gc.collect()

# 2. Clear PyTorch's cache
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    
# 3. Double-check available memory (Optional but helpful)
free_vram = torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0)
print(f"Available VRAM: {free_vram / 1024**3:.2f} GB")

Available VRAM: 8.26 GB


In [3]:
!pip install trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.6/721.6 kB 10.9 MB/s eta 0:00:00 0:00:01


In [2]:
# The % prefix forces it into the active notebook kernel
%pip install transformers huggingface_hub peft trl datasets bitsandbytes accelerate --no-cache-dir

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.6/721.6 kB 3.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 85.4 MB/s eta 0:00:00a 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [2]:
import bitsandbytes as bnb
print(bnb.__version__) # This should now print 0.46.1 or higher

ModuleNotFoundError: No module named 'bitsandbytes'

In [3]:
import torch
import gc
import os
from transformers import AutoTokenizer, AutoModelForCausalLM, EarlyStoppingCallback
from peft import get_peft_model, LoraConfig
from trl import SFTTrainer, SFTConfig

# 0. Tear off the eyepatch: Let PyTorch see BOTH Kaggle GPUs again
if "CUDA_VISIBLE_DEVICES" in os.environ:
    del os.environ["CUDA_VISIBLE_DEVICES"]

# 1. Total VRAM Wipe
gc.collect()
torch.cuda.empty_cache()

print(f"GPUs available: {torch.cuda.device_count()}") # Should say 2!

# 2. Dataset Split (Phase 2 feature)
dataset_split = dataset.train_test_split(test_size=0.1)
train_data = dataset_split["train"]
eval_data = dataset_split["test"]

model_id = "Qwen/CodeQwen1.5-7B-Chat"

# 3. Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

# 4. PURE FLOAT16 LOAD (Distributed across both GPUs - Your method)
print("Loading pure FP16 model across multiple GPUs...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float16,  
    trust_remote_code=True
)
model.config.torch_dtype = torch.float16

# 5. Apply LoRA (Phase 2 Settings: higher dropout for regularization)
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.15, 
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, peft_config)

# 6. Formatting Function (Phase 2 Master Plan)
def formatting_prompts_func(example):
    return (
        f"<|im_start|>system\n"
        f"You are a SQL expert. Strictly output ONLY the SQL query requested. "
        f"Do NOT include additional columns, filters, or JOINs not explicitly required by the question.<|im_end|>\n"
        f"<|im_start|>user\n"
        f"Context: {example['context']}\n\n"
        f"Question: {example['question']}<|im_end|>\n"
        f"<|im_start|>assistant\n"
        f"{example['query']}<|im_end|>"
    )

# 7. SFTConfig (distributed config + Phase 2 Eval mechanics)
sft_config = SFTConfig(
    output_dir="./codeqwen-7b-sql-v2",
    max_length=512,                  
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-5,
    max_steps=1200,
    logging_steps=10,
    eval_strategy="steps",           
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    load_best_model_at_end=True,
    fp16=True,                       
    bf16=False,     
    optim="paged_adamw_8bit",        
    gradient_checkpointing=True,     
    report_to="none"
)

# 8. Launch Trainer (Cleaned up!)
trainer = SFTTrainer(
    model=model,
    train_dataset=train_data,
    eval_dataset=eval_data,
    formatting_func=formatting_prompts_func, 
    args=sft_config,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

print("🚀 Launching Hybrid Training: Phase 2 Logic on your stable architecture...")
trainer.train()

GPUs available: 2


config.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/972 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/1.42M [00:00<?, ?B/s]

...

Tokenizing eval dataset:   0%|          | 0/700 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 4, 'pad_token_id': 92298}.


🚀 Launching Hybrid Training: Phase 2 Logic on your stable architecture...


Step,Training Loss,Validation Loss
200,0.648717,0.646980
400,0.527970,0.509117
600,0.442044,0.441825
800,0.402745,0.418217
1000,0.391643,0.406216
1200,0.384093,0.400943


TrainOutput(global_step=1200, training_loss=0.5396683410803477, metrics={'train_runtime': 7230.0218, 'train_samples_per_second': 1.328, 'train_steps_per_second': 0.166, 'total_flos': 5.874621554515968e+16, 'train_loss': 0.5396683410803477})

In [ ]:
import shutil

# 1. Define where to save the adapters
save_directory = "./final-sql-adapters"

# 2. Save the trained LoRA adapters AND the tokenizer
print("Saving LoRA adapters and tokenizer...")
trainer.save_model(save_directory)
tokenizer.save_pretrained(save_directory)

# 3. Compress the folder into a single ZIP file
print("Zipping the folder for evacuation...")
shutil.make_archive("codeqwen-sql-victory", 'zip', save_directory)

print("Evacuation ready! Download 'codeqwen-sql-victory.zip' from the right-hand panel.")

# Precision Inference

We use a Hardened System Prompt to provide an explicit negative constraint. This acts as a logical firewall against the model's tendency to add "extra" filters.

In [11]:
def precision_generate_sql(question, context):
    # The Zero-Tolerance System Prompt
    system_instruction = (
        "You are a SQL expert. Strictly output ONLY the SQL query requested. "
        "Do NOT include additional columns, filters, or JOINs not explicitly required by the question. "
        "Ignore all DDL columns that do not directly answer the user prompt."
    )
    
    messages = [
        {"role": "system", "content": system_instruction},
        {"role": "user", "content": f"Context: {context}\n\nQuestion: {question}"}
    ]
    
    prompt = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs, 
            max_new_tokens=128,
            do_sample=False,
            # Ensure proper EOS handling for Qwen
            eos_token_id=tokenizer.convert_tokens_to_ids("<|im_end|>"),
            pad_token_id=tokenizer.eos_token_id
        )
    
    generated_ids = outputs[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

# # Test the new precision logic
# test_q = "List the vote ids and phone numbers of all votes."
# test_ddl = "CREATE TABLE votes (vote_id NUMBER, phone_number TEXT, state TEXT) CREATE TABLE distractors (id NUMBER, name TEXT)"
# print(f"Precision Result: {precision_generate_sql(test_q, test_ddl)}")

In [8]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

# 1. Load the Base Model (FP16, distributed across GPUs)
model_id = "Qwen/CodeQwen1.5-7B-Chat"
print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float16,
)

# 2. ATTACH YOUR NEWLY TRAINED ADAPTERS 
adapter_path = "./final-sql-adapters"  # The folder we saved to earlier
print(f"Attaching fine-tuned adapters from {adapter_path}...")
model = PeftModel.from_pretrained(base_model, adapter_path)

# 3. Load the Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

print(" Fine-tuned Precision Model Loaded! You can now run the inference cell.")

Loading base model...


config.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

...

Attaching fine-tuned adapters from ./final-sql-adapters...


tokenizer_config.json:   0%|          | 0.00/972 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/1.42M [00:00<?, ?B/s]

 Fine-tuned Precision Model Loaded! You can now run the inference cell.


In [ ]:
import re

# 1. SQL Normalizer Function
def normalize_sql(sql):
    """Removes extra spaces, newlines, trailing semicolons, and lowercases the query."""
    sql = re.sub(r'\s+', ' ', sql).strip().lower()
    sql = sql.rstrip(';')
    return sql

# 2. Metric Calculator Function
def calculate_metrics(predicted, truth):
    pred_norm = normalize_sql(predicted)
    truth_norm = normalize_sql(truth)
    
    # --- Exact Match (EM) ---
    exact_match = int(pred_norm == truth_norm)
    
    # --- Proximal Match (Token Overlap) ---
    # We split the SQL into words/tokens and compare the sets.
    pred_tokens = set(pred_norm.split())
    truth_tokens = set(truth_norm.split())
    
    intersection = pred_tokens.intersection(truth_tokens)
    union = pred_tokens.union(truth_tokens)
    
    # Jaccard Similarity: (Overlap / Total Unique Tokens)
    jaccard_score = len(intersection) / len(union) if union else 0
    
    # If the queries share 80% or more of the same keywords/columns, we count it as a Proximal Match
    proximal_match = int(jaccard_score >= 0.80)
    
    return exact_match, proximal_match

# 3. Run the Evaluation
# Change sample_size if you want to test more, but keep it low to save Kaggle time!
sample_size = 15 
samples = eval_data.shuffle(seed=42).select(range(sample_size))

em_total = 0
prox_total = 0

print(f"🔍 Starting Evaluation on {sample_size} random samples...\n")

for i, row in enumerate(samples):
    question = row['question']
    context = row['context']
    
    # 🔴 THE FIX: 'answer' matches the dataset schema
    ground_truth = row['answer'] 
    
    # Use your newly loaded fine-tuned model
    prediction = precision_generate_sql(question, context)
    
    em, prox = calculate_metrics(prediction, ground_truth)
    em_total += em
    prox_total += prox
    
    print(f"--- Sample {i+1} ---")
    print(f"Question:  {question}")
    print(f"Truth:     {ground_truth}")
    print(f"Predicted: {prediction}")
    print(f"Metrics:   EM={em} | Proximal={prox}\n")

# 4. Final Report
print("==============================")
print(" FINAL EVALUATION RESULTS ")
print("==============================")
print(f"Exact Match (EM): {(em_total/sample_size)*100:.1f}%")
print(f"Proximal Match:   {(prox_total/sample_size)*100:.1f}%")
print("==============================")

The following generation flags are not valid and may be ignored: ['top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


🔍 Starting Evaluation on 15 random samples...

--- Sample 1 ---
Question:  Which Record has a Location/Attendance of palace of auburn hills 8,108?
Truth:     SELECT record FROM table_name_22 WHERE location_attendance = "palace of auburn hills 8,108"
Predicted: SELECT * FROM records WHERE location_attendance  =  "Palace of Auburn Hills 8,108"
Metrics:   EM=0 | Proximal=0

--- Sample 2 ---
Question:  What is the highest number of silver medals of the team with less than 0 bronzes?
Truth:     SELECT MAX(silver) FROM table_name_85 WHERE bronze < 0
Predicted: SELECT max(silver) FROM team WHERE bronze  <  0
Metrics:   EM=0 | Proximal=0

--- Sample 3 ---
Question:  How did the game number 50 end?
Truth:     SELECT score FROM table_23248940_9 WHERE game = 50
Predicted: SELECT end_how FROM game WHERE game_id  =  50
Metrics:   EM=0 | Proximal=0

--- Sample 4 ---
Question:  Name the california where alaska tennessee
Truth:     SELECT california FROM table_17425749_1 WHERE alaska = "Tennessee"
Pre